RNN:由一个神经元接收输入，产生输出并将该输出反送回自身组成。在每个时间步长t，该循环神经元接受输入 $x_{(t)}$ 和前一个时间步长 $y_{(t-1)}$ 的输出。由于在第一个时间步长中没有先前的输出，因此通常将其设置为0。

单个实例的循环层的输出：

$y_{(t)}=\phi (W^{T}_{x}x_{(t)}+W^{T}_{y}y_{(t-1)}+b)$

可以通过将时间步长 $t$ 处的所有输入都放在输入矩阵 $X_{(t)}$ 中，来一次性地计算整个小批量中的递归层的输出.

小批量中的所有实例的循环神经元层的输出：

$$
\begin{aligned}
Y_{(t)} &= \phi\left( X_{(t)} W_x + Y_{(t-1)} W_y + \mathbf{b} \right) \\
        &= \phi\left( \big[ X_{(t)} \; Y_{(t-1)} \big] W + \mathbf{b} \right)
\quad \text{其中} \quad
W = \begin{bmatrix} W_x \\ W_y \end{bmatrix}
\end{aligned}
$$

在此等式中：

- $Y_{(t)}$ 是一个 $m \times n_{\text{neurons}}$ 矩阵，包含小批量处理中每个实例在时间步长 $t$ 时该层的输出（$m$ 是小批量处理中的实例数量，$n_{\text{neurons}}$ 是神经元数量）。

- $X_{(t)}$ 是一个 $m \times n_{\text{inputs}}$ 矩阵，包含所有实例的输入（$n_{\text{inputs}}$ 是输入特征的数量）。

- $W_x$ 是一个 $n_{\text{inputs}} \times n_{\text{neurons}}$ 矩阵，包含当前时间步长的输入的连接权重。

- $W_y$ 是一个 $n_{\text{neurons}} \times n_{\text{neurons}}$ 矩阵，包含前一个时间步长的输出的连接权重。

- $\mathbf{b}$ 是大小为 $n_{\text{neurons}}$ 的向量，包含每个神经元的偏置项。

- 权重矩阵 $W_x$ 和 $W_y$ 经常垂直合并成形状为 $(n_{\text{inputs}} + n_{\text{neurons}}) \times n_{\text{neurons}}$ 的单个权重矩阵 $W$（见公式 15–2 的第二行）。

- 符号 $[X_{(t)} \; Y_{(t-1)}]$ 表示矩阵 $X_{(t)}$ 和 $Y_{(t-1)}$ 的**水平合并**（即列拼接）。

In [ ]:
#预测时间序列

def generate_time_series(batch_size, n_steps): 
    freq1, freq2, offsets1, offsets2 = np.random.rand(4, batch_size, 1) 
    time = np.linspace(0, 1, n_steps) 
    series = 0.5 * np.sin((time - offsets1) * (freq1 * 10 + 10))  #   wave 1 
    series += 0.2 * np.sin((time - offsets2) * (freq2 * 20 + 20)) # + wave 2 
    series += 0.1 * (np.random.rand(batch_size, n_steps) - 0.5)   # + noise 
    return series[..., np.newaxis].astype(np.float32) 

n_steps = 50 
series = generate_time_series(10000, n_steps + 1) 
X_train, y_train = series[:7000, :n_steps], series[:7000, -1] 
X_valid, y_valid = series[7000:9000, :n_steps], series[7000:9000, -1] 
X_test, y_test = series[9000:, :n_steps], series[9000:, -1] 

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[50, 1]),
    keras.layers.Dense(1)
])

In [ ]:
#深度RNN

model = keras.models.Sequential([ 
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]), 
    keras.layers.SimpleRNN(20, return_sequences=True), 
    keras.layers.SimpleRNN(1) 
]) 

model = keras.models.Sequential([ 
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]), 
    keras.layers.SimpleRNN(20), 
    keras.layers.Dense(1) 
]) 

In [ ]:
#预测未来几个时间步长

series = generate_time_series(1, n_steps + 10) 
X_new, Y_new = series[:, :n_steps], series[:, n_steps:] 
X = X_new 
for step_ahead in range(10): 
    y_pred_one = model.predict(X[:, step_ahead:])[:, np.newaxis, :] 
    X = np.concatenate([X, y_pred_one], axis=1) 
Y_pred = X[:, n_steps:] 

model = keras.models.Sequential([ 
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]), 
    keras.layers.SimpleRNN(20), 
    keras.layers.Dense(10) 
]) 
Y_pred = model.predict(X_new) 

Y = np.empty((10000, n_steps, 10)) # each target is a sequence of 10D vectors 
for step_ahead in range(1, 10 + 1): 
    Y[:, :, step_ahead - 1] = series[:, step_ahead:step_ahead + n_steps, 0] 
Y_train = Y[:7000] 
Y_valid = Y[7000:9000] 
Y_test = Y[9000:] 

model = keras.models.Sequential([ 
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]), 
    keras.layers.SimpleRNN(20, return_sequences=True), 
    keras.layers.TimeDistributed(keras.layers.Dense(10)) 
]) 

def last_time_step_mse(Y_true, Y_pred): 
    return keras.metrics.mean_squared_error(Y_true[:, -1], Y_pred[:, -1]) 
optimizer = keras.optimizers.Adam(lr=0.01) 
model.compile(loss="mse", optimizer=optimizer, metrics=[last_time_step_mse]) 


In [ ]:
#处理长序列

class LNSimpleRNNCell(keras.layers.Layer): 
    def __init__(self, units, activation="tanh", kwargs): 
        super().__init__( kwargs) 
        self.state_size = units 
        self.output_size = units 
        self.simple_rnn_cell = keras.layers.SimpleRNNCell(units, 
                                                          activation=None) 
        self.layer_norm = keras.layers.LayerNormalization() 
        self.activation = keras.activations.get(activation) 
    def call(self, inputs, states): 
        outputs, new_states = self.simple_rnn_cell(inputs, states) 
        norm_outputs = self.activation(self.layer_norm(outputs)) 
        return norm_outputs, [norm_outputs] 

model = keras.models.Sequential([ 
    keras.layers.RNN(LNSimpleRNNCell(20), return_sequences=True, 
                     input_shape=[None, 1]), 
    keras.layers.RNN(LNSimpleRNNCell(20), return_sequences=True), 
    keras.layers.TimeDistributed(keras.layers.Dense(10)) 
])

In [ ]:
#LSTM单元
model = keras.models.Sequential([ 
    keras.layers.LSTM(20, return_sequences=True, input_shape=[None, 1]), 
    keras.layers.LSTM(20, return_sequences=True), 
    keras.layers.TimeDistributed(keras.layers.Dense(10)) 
])

model = keras.models.Sequential([ 
    keras.layers.RNN(keras.layers.LSTMCell(20), return_sequences=True, 
                     input_shape=[None, 1]), 
    keras.layers.RNN(keras.layers.LSTMCell(20), return_sequences=True), 
    keras.layers.TimeDistributed(keras.layers.Dense(10)) 
])

LSTM计算公式：

$$
\begin{aligned}
i_{(t)} &= \sigma\!\left( W_{xi}^\top x_{(t)} + W_{hi}^\top h_{(t-1)} + b_i \right) \\
f_{(t)} &= \sigma\!\left( W_{xf}^\top x_{(t)} + W_{hf}^\top h_{(t-1)} + b_f \right) \\
o_{(t)} &= \sigma\!\left( W_{xo}^\top x_{(t)} + W_{ho}^\top h_{(t-1)} + b_o \right) \\
g_{(t)} &= \tanh\!\left( W_{xg}^\top x_{(t)} + W_{hg}^\top h_{(t-1)} + b_g \right) \\
c_{(t)} &= f_{(t)} \odot c_{(t-1)} + i_{(t)} \odot g_{(t)} \\
y_{(t)} &= h_{(t)} = o_{(t)} \odot \tanh\!\left( c_{(t)} \right)
\end{aligned}
$$

在此等式中：

- $W_{xi}$、$W_{xf}$、$W_{xo}$、$W_{xg}$ 是四层中的每层与输入向量 $x_{(t)}$ 连接的权重矩阵。

- $W_{hi}$、$W_{hf}$、$W_{ho}$ 和 $W_{hg}$ 是四层中的每层与先前的短期状态 $h_{(t-1)}$ 连接的权重矩阵。

- $b_i$、$b_f$、$b_o$ 和 $b_g$ 是四层中每层的偏置项。  
  **注意**：TensorFlow 将 $b_f$ 初始化为一个全为 1（不是 0）的向量。这样可以防止在训练开始时“忘记一切”（即避免初始遗忘门恒为 0，从而保留长期记忆）。

GRU单元计算公式：

$$
\begin{aligned}
z_{(t)} &= \sigma\!\left( W_{xz}^\top x_{(t)} + W_{hz}^\top h_{(t-1)} + b_z \right) \\
r_{(t)} &= \sigma\!\left( W_{xr}^\top x_{(t)} + W_{hr}^\top h_{(t-1)} + b_r \right) \\
g_{(t)} &= \tanh\!\left( W_{xg}^\top x_{(t)} + W_{hg}^\top \big(r_{(t)} \odot h_{(t-1)}\big) + b_g \right) \\
h_{(t)} &= z_{(t)} \odot h_{(t-1)} + \big(1 - z_{(t)}\big) \odot g_{(t)}
\end{aligned}
$$

In [ ]:
#使用一维卷积层处理序列

model = keras.models.Sequential([ 
    keras.layers.Conv1D(filters=20, kernel_size=4, strides=2, padding="valid", 
                        input_shape=[None, 1]), 
    keras.layers.GR (20, return_sequences=True), 
    keras.layers.GR (20, return_sequences=True), 
    keras.layers.TimeDistributed(keras.layers.Dense(10)) 
]) 
model.compile(loss="mse", optimizer="adam", metrics=[last_time_step_mse]) 
history = model.fit(X_train, Y_train[:, 3::2], epochs=20, 
                    validation_data=(X_valid, Y_valid[:, 3::2])) 

In [ ]:
#WaveNet

model = keras.models.Sequential() 
model.add(keras.layers.InputLayer(input_shape=[None, 1])) 
for rate in (1, 2, 4, 8)  2: 
    model.add(keras.layers.Conv1D(filters=20, kernel_size=2, padding="causal", 
                                  activation="relu", dilation_rate=rate)) 
model.add(keras.layers.Conv1D(filters=10, kernel_size=1)) 
model.compile(loss="mse", optimizer="adam", metrics=[last_time_step_mse]) 
history = model.fit(X_train, Y_train, epochs=20, 
                    validation_data=(X_valid, Y_valid)) 